# Phase 5 — Tool Use: Build the Reminder Bot

Until now, Claude could only generate text. This phase teaches it to **do things** — fetch the current time, calculate a future date, set a reminder. The same pattern powers every real-world Claude integration: weather bots, GitHub assistants, Claude Code itself.

## What you'll learn
1. **Tool function** — plain Python function Claude can invoke
2. **Tool schema** — JSON description that tells Claude when/how to call it
3. **The tool loop** — the single most important pattern in this course
4. **Multi-tool chains** — Claude using several tools to solve one query

Goal: build a reminder bot. Ask *"remind me about my doctor's appointment a week from Thursday"* and watch it chain three tools to handle it.

## Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

import json
from datetime import datetime, timedelta
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

## Step 1 — Define a tool *function*

A tool function is just a plain Python function. Best practices:
- Descriptive name and arg names (Claude reads these)
- Validate inputs and raise `ValueError` with a helpful message — Claude *sees* errors and retries with fixed arguments

In [ ]:
def get_current_datetime(date_format: str = "%Y-%m-%d %H:%M:%S") -> str:
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

# Test it yourself first
print(get_current_datetime())

## Step 2 — Define a tool *schema*

The schema is a JSON description Claude uses to decide whether to call the function. Three required pieces:
- `name` — must match how you'll dispatch it later
- `description` — 3-4 sentences: what it does, when to use it, what it returns
- `input_schema` — JSON Schema describing the arguments

In [ ]:
get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": (
        "Returns the current date and time as a formatted string. "
        "Use this whenever the user asks about 'today', 'now', 'this week', or any relative date "
        "that requires knowing the current moment. The format argument controls the output shape."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A strftime-compatible format string, e.g. '%Y-%m-%d' for date only."
            }
        },
        "required": ["date_format"]
    }
}

## Step 3 — Single round-trip: see Claude *request* a tool

Send a request with `tools=[...]`. If Claude decides it needs a tool, the response will have `stop_reason='tool_use'` and a `tool_use` block in `content` instead of (or alongside) a text block.

In [ ]:
messages = [{"role": "user", "content": "What's today's date?"}]

response = client.messages.create(
    model=model,
    max_tokens=500,
    tools=[get_current_datetime_schema],
    messages=messages,
)

print("Stop reason:", response.stop_reason)
print("Content blocks:")
for block in response.content:
    print(" ", block)

You should see `stop_reason='tool_use'` and a `ToolUseBlock` with `name='get_current_datetime'` and an `input` dict. Claude is asking *you* to run this function.

## Step 4 — Send back the tool *result*

Three things happen here, all critical:

1. **Append Claude's whole reply to history** (the `tool_use` block included)
2. **Run the function with the arguments Claude requested**
3. **Append a `tool_result` block as a USER message** (yes, user — even though you're returning a tool's output)

The `tool_use_id` is what links the result back to the original request. Crucial when there are multiple tool calls in one turn.

In [ ]:
# 1. Append Claude's response to history (full content, not just text!)
messages.append({"role": "assistant", "content": response.content})

# 2. Find the tool_use block and run the function
tool_use_block = next(b for b in response.content if b.type == "tool_use")
tool_result_text = get_current_datetime(**tool_use_block.input)
print("Tool returned:", tool_result_text)

# 3. Append the result as a user message
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": tool_use_block.id,
        "content": tool_result_text,
    }]
})

# 4. Call Claude again with the updated history. It should now give a text answer.
response = client.messages.create(
    model=model,
    max_tokens=500,
    tools=[get_current_datetime_schema],   # ← STILL pass the tools, even when not expecting another call
    messages=messages,
)

print("\nStop reason:", response.stop_reason)
print("Final answer:", response.content[0].text)

**That's the whole tool-use pattern in 4 steps.** Everything else in this notebook is just generalizing this into a reusable loop and adding more tools.

## Step 5 — The reusable loop: `run_conversation`

In real use you don't know how many tools Claude will need. Maybe one, maybe five chained. The solution: a `while` loop that keeps calling Claude until `stop_reason != 'tool_use'`.

We'll write three helper functions:
- `run_tool(name, input)` — dispatcher: matches tool name to Python function
- `run_tools(message)` — extracts all tool_use blocks from a message, runs them, returns `tool_result` blocks
- `run_conversation(messages, tools)` — the main loop

In [ ]:
def run_tool(tool_name: str, tool_input: dict):
    """Dispatch: tool name → Python function."""
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    # (more tools added below)
    raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(message) -> list:
    """Run every tool_use block in an assistant message; return matching tool_result blocks."""
    tool_use_blocks = [b for b in message.content if b.type == "tool_use"]
    results = []
    for block in tool_use_blocks:
        try:
            output = run_tool(block.name, block.input)
            results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": json.dumps(output) if not isinstance(output, str) else output,
                "is_error": False,
            })
        except Exception as e:
            results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": f"Error: {e}",
                "is_error": True,
            })
    return results


def run_conversation(messages: list, tools: list, verbose: bool = True) -> str:
    """Keep calling Claude until it stops requesting tools. Returns the final text reply."""
    while True:
        response = client.messages.create(
            model=model,
            max_tokens=1000,
            tools=tools,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            # Final text reply — extract and return
            text_blocks = [b.text for b in response.content if b.type == "text"]
            return "\n".join(text_blocks)

        # Otherwise: run tools, append results, loop
        if verbose:
            for b in response.content:
                if b.type == "tool_use":
                    print(f"  → calling {b.name}({b.input})")

        tool_results = run_tools(response)
        messages.append({"role": "user", "content": tool_results})

In [ ]:
# Try the loop with just one tool
messages = [{"role": "user", "content": "What's today's date and what day of the week is it?"}]
answer = run_conversation(messages, tools=[get_current_datetime_schema])
print("\nFinal answer:\n", answer)

## Step 6 — Add two more tools

To handle *"remind me a week from Thursday,"* Claude needs:
1. **`get_current_datetime`** — already have it
2. **`add_duration_to_datetime`** — Claude is bad at arithmetic on dates
3. **`set_reminder`** — actually "set" the reminder (we'll just print, like a fake DB write)

In [ ]:
# Tool 2: date arithmetic
def add_duration_to_datetime(base_datetime: str, days: int = 0, hours: int = 0) -> str:
    """Add days/hours to an ISO-like datetime string."""
    dt = datetime.strptime(base_datetime, "%Y-%m-%d %H:%M:%S")
    result = dt + timedelta(days=days, hours=hours)
    return result.strftime("%Y-%m-%d %H:%M:%S (%A)")

add_duration_schema = {
    "name": "add_duration_to_datetime",
    "description": (
        "Adds a duration to a base datetime and returns the resulting datetime. "
        "Use this for any date math — 'next week', 'in 3 days', '2 hours from now'. "
        "The base_datetime must be in '%Y-%m-%d %H:%M:%S' format."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "base_datetime": {"type": "string", "description": "Starting datetime, e.g. '2026-05-27 14:30:00'"},
            "days": {"type": "integer", "description": "Number of days to add (can be negative)"},
            "hours": {"type": "integer", "description": "Number of hours to add"}
        },
        "required": ["base_datetime"]
    }
}

# Tool 3: "set" a reminder (mock — just store in a list)
REMINDERS = []

def set_reminder(message: str, when: str) -> str:
    REMINDERS.append({"message": message, "when": when})
    return f"Reminder set: '{message}' at {when}"

set_reminder_schema = {
    "name": "set_reminder",
    "description": (
        "Stores a reminder with a message and a target datetime. "
        "Call this only AFTER you know the exact datetime — use the datetime tools first if needed."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "message": {"type": "string", "description": "What to remind the user about"},
            "when": {"type": "string", "description": "Target datetime, e.g. '2026-06-04 09:00:00'"}
        },
        "required": ["message", "when"]
    }
}

# Extend the dispatcher
def run_tool(tool_name: str, tool_input: dict):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    if tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    if tool_name == "set_reminder":
        return set_reminder(**tool_input)
    raise ValueError(f"Unknown tool: {tool_name}")

ALL_TOOLS = [get_current_datetime_schema, add_duration_schema, set_reminder_schema]
print("3 tools ready.")

## Step 7 — The full chain: "remind me a week from Thursday"

Watch the verbose output. Claude should:
1. Call `get_current_datetime` (to know what "Thursday" means relative to now)
2. Call `add_duration_to_datetime` (to compute "a week after that Thursday")
3. Call `set_reminder` (with the computed target time)
4. Then give a final text reply confirming

In [ ]:
REMINDERS.clear()
messages = [{
    "role": "user",
    "content": "Please remind me about my doctor's appointment a week from Thursday at 9am."
}]

answer = run_conversation(messages, tools=ALL_TOOLS)

print("\nFinal answer:\n", answer)
print("\nReminders stored:", REMINDERS)

**Stop and appreciate what just happened.** You wrote three short Python functions. Claude figured out — by itself — that it needed all three, in the right order, with the right arguments derived from a vague natural-language request. That's the entire promise of tool use in one example.

## Step 8 — Bonus: try a longer query

Same loop, a more interesting request. Notice you never had to change `run_conversation` — it handles arbitrary tool chains automatically.

In [ ]:
REMINDERS.clear()
messages = [{
    "role": "user",
    "content": (
        "Set two reminders for me: one to call my dentist in 3 days at 10am, "
        "and one to pay the rent in 2 weeks at 9am."
    )
}]

answer = run_conversation(messages, tools=ALL_TOOLS)
print("\nFinal answer:\n", answer)
print("\nReminders stored:")
for r in REMINDERS:
    print(" ", r)

## Mini-exercise

1. **Add a `list_reminders` tool** that returns the current `REMINDERS` list. After setting some, ask *"what reminders do I have?"* — Claude should call your new tool and summarize.
2. **Make Claude self-correct.** Pass a bad datetime to `add_duration_to_datetime` (e.g. patch the function to raise `ValueError("bad format")` if the year is `2099`). Then ask Claude something that would produce 2099. Watch the error come back in a `tool_result` block with `is_error=True` — Claude should *retry* with corrected input.
3. **Inspect the message history.** After step 7 completes, `print(len(messages))` and look at the roles in order. You'll see the rhythm: user → assistant(tool_use) → user(tool_result) → assistant(tool_use) → user(tool_result) → … → assistant(text).

Tell me what you saw, and we'll head to **Phase 6: prompt caching** — where we make this same bot 50%+ cheaper by caching the tool schemas and system prompt.